# 16. Aturan Keputusan dan Kalibrasi di Dalam Lipatan

Notebook 15 menilai semua konfigurasi dengan `argmax`. Namun aplikasi tidak memakai `argmax` saja: ia memakai **aturan keputusan** untuk kelas Prediabetes. Notebook ini menjawab tiga pertanyaan yang masih terbuka, **tanpa melatih ulang model apa pun** (hanya memakai probabilitas lipatan luar yang sudah tersimpan oleh notebook 15):

1. **Aturan mana yang dipakai?** `argmax`, threshold (`P(kelas 1) >= t`, yang dipakai `predict.py`), atau pengali alpha (`P(kelas 1) x alpha` lalu `argmax`, yang tertulis di proposal).
2. **Apakah kalibrasi membantu?** (Platt/vector scaling pada log-probabilitas), diuji dengan atau tanpa.
3. **Konfigurasi mana yang layak jadi model final?** Dipilih dengan aturan satu galat baku (dijelaskan di bagian 8).

**Mengapa "di dalam lipatan":** pada notebook 12 threshold dicari pada seluruh OOF lalu dinilai pada OOF yang sama, sehingga hasilnya optimistis. Di sini, untuk setiap lipatan luar *k*, threshold atau alpha (dan kalibrator) **dicari hanya dari lipatan lain**, lalu diterapkan pada lipatan *k* yang tidak ikut menentukannya. Skor per lipatan ini adalah estimasi yang lebih jujur.

**Keterbatasan yang perlu ditulis:** probabilitas lipatan lain berasal dari model yang dilatih pada data yang mencakup lipatan *k*. Pengaruhnya pada satu angka skalar (threshold atau alpha) kecil, tetapi bukan nol.

**Keamanan:**
- Hanya membaca folder `ckpt` dari notebook 15. `train_final.csv` dan `test_final_locked.csv` tidak dibaca.
- Tidak menulis ke `artifacts` dan `data`. Keluaran hanya ke `hasil_eksperimen/16_aturan_keputusan`.

**Keluaran:**
- `metrik_per_lipatan_aturan.csv`, `ringkasan_aturan_mean_sd.csv`
- `ranking_kandidat.csv`: seluruh kombinasi (skenario, model, kalibrasi, aturan) berurutan
- `stabilitas_parameter.csv`: threshold dan alpha yang terpilih di tiap lipatan (jika berubah-ubah, aturannya tidak stabil)
- `uji_aturan_berpasangan.csv`
- `kandidat_final.json`: usulan konfigurasi untuk notebook 17 (**usulan, bukan keputusan**)
- `grafik_aturan.png`

## 0. Mengatur Folder Kerja

Notebook ini disimpan di folder `notebooks`, sedangkan data berada di folder `glysense_backend`. Sel berikut memindahkan folder kerja satu tingkat ke atas. Aman dijalankan berulang.

In [ ]:
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print(os.getcwd())

## 1. Pengaturan

In [ ]:
import glob
import json
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")

UJI_CEPAT = False             # True = membaca hasil uji cepat notebook 15, BUKAN untuk penelitian

KRITERIA_PILIH = "G_Mean"     # kriteria utama memilih kandidat: "G_Mean" atau "Macro_F1"
SYARAT_RECALL_1 = 0.0         # opsional: kandidat dengan rata-rata Recall Prediabetes di bawah ini dibuang (mis. 0.30)

INPUT_DIR = "hasil_eksperimen/15_eksperimen_utama"
OUTPUT_DIR = "hasil_eksperimen/16_aturan_keputusan"
if UJI_CEPAT:
    INPUT_DIR = "hasil_eksperimen/15_uji_cepat"
    OUTPUT_DIR = "hasil_eksperimen/16_uji_cepat"

CKPT_DIR = os.path.join(INPUT_DIR, "ckpt")
assert os.path.isdir(CKPT_DIR), "Folder ckpt notebook 15 tidak ditemukan. Jalankan notebook 15 dulu."
assert "artifacts" not in OUTPUT_DIR and not OUTPUT_DIR.startswith("data"), "Folder output tidak boleh di artifacts/data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Mode uji cepat :", UJI_CEPAT)
print("Membaca        :", CKPT_DIR)
print("Menulis        :", OUTPUT_DIR)

## 2. Memuat Probabilitas Lipatan Luar dari Notebook 15

Tiap file `seed*_fold*_[A/B].npz` berisi probabilitas validasi luar untuk RF, XGBoost, dan dua varian Hybrid. Hanya seed dengan **seluruh lipatan lengkap** untuk suatu skenario yang dipakai.

In [ ]:
pola = re.compile(r"seed(\d+)_fold(\d+)_([AB])\.npz$")
mentah = {}
for p in sorted(glob.glob(os.path.join(CKPT_DIR, "*.npz"))):
    m = pola.search(os.path.basename(p))
    if not m:
        continue
    seed, fold, sk = int(m.group(1)), int(m.group(2)), m.group(3)
    d = np.load(p)
    mentah.setdefault((seed, sk), {})[fold] = {k: d[k] for k in ["y", "rf", "xgb", "hyb_nobal", "hyb_bal"]}

assert mentah, "Tidak ada file ckpt. Jalankan notebook 15 dulu."
N_OUTER = max(f for g in mentah.values() for f in g)

GRUP = {}
for kunci_grup, folds in mentah.items():
    if sorted(folds) == list(range(1, N_OUTER + 1)):
        GRUP[kunci_grup] = folds
    else:
        print("PERINGATAN: seed/skenario", kunci_grup, "belum lengkap, dilewati.")

assert GRUP, "Tidak ada seed dengan lipatan lengkap."
print("N lipatan luar:", N_OUTER)
print("Grup lengkap (seed, skenario):", sorted(GRUP))

## 3. Aturan Keputusan, Kalibrasi, dan Metrik

| Aturan | Cara memutuskan | Parameter dicari dengan |
|---|---|---|
| `argmax` | kelas dengan probabilitas terbesar | tidak ada |
| `thr_gmean` | `P1 >= t` maka kelas 1, selain itu pilih 0 atau 2 menurut probabilitas | G-Mean, lalu Recall kelas 1, lalu Macro-F1 (sama dengan notebook 12 dan `predict.py`) |
| `thr_macrof1` | sama, tetapi kriteria Macro-F1 | Macro-F1, lalu G-Mean, lalu Recall kelas 1 |
| `alpha_macrof1` | `argmax` dari `[P0, alpha x P1, P2]` | Macro-F1 (rumusan proposal) |
| `alpha_gmean` | sama, tetapi kriteria G-Mean | G-Mean |

Tiap aturan dua kali dinilai: tanpa kalibrasi dan dengan kalibrasi (regresi logistik multinomial pada log-probabilitas, dilatih dari lipatan lain). Kriteria sengaja dipasangkan di kedua keluarga aturan supaya yang dibandingkan adalah **aturannya**, bukan kriterianya.

In [ ]:
def metrik_dari_cm(cm):
    tp = np.diag(cm).astype(float)
    benar = cm.sum(axis=1).astype(float)
    pred = cm.sum(axis=0).astype(float)
    rec = np.divide(tp, benar, out=np.zeros(3), where=benar > 0)
    pre = np.divide(tp, pred, out=np.zeros(3), where=pred > 0)
    f1 = np.divide(2 * pre * rec, pre + rec, out=np.zeros(3), where=(pre + rec) > 0)
    return rec, pre, f1, float(np.prod(rec) ** (1 / 3)), float(f1.mean()), float(tp.sum() / cm.sum())


def cm_dari(y, pred):
    return np.bincount(y * 3 + pred, minlength=9).reshape(3, 3)


GRID_THR = np.unique(np.concatenate([np.arange(0.002, 0.05, 0.002), np.arange(0.05, 0.96, 0.01)]))
GRID_ALPHA = np.unique(np.round(np.concatenate([[1.0], np.geomspace(1.0, 200.0, 80)]), 3))
RULES = ["argmax", "thr_gmean", "thr_macrof1", "alpha_macrof1", "alpha_gmean"]


def norm(P):
    return P / P.sum(axis=1, keepdims=True)


def prediksi(P, aturan, param):
    if aturan == "argmax":
        return P.argmax(axis=1)
    if aturan.startswith("thr"):
        arg02 = np.where(P[:, 0] >= P[:, 2], 0, 2)
        return np.where(P[:, 1] >= param, 1, arg02)
    return (P * np.array([1.0, param, 1.0])).argmax(axis=1)


def skor(cm, aturan):
    rec, pre, f1, gm, mf1, acc = metrik_dari_cm(cm)
    if aturan.endswith("macrof1"):
        return (mf1, gm, rec[1])
    return (gm, rec[1], mf1)


def cari_param(P, y, aturan):
    if aturan == "argmax":
        return None
    grid = GRID_THR if aturan.startswith("thr") else GRID_ALPHA
    terbaik, skor_terbaik = None, (-1.0, -1.0, -1.0)
    for g in grid:
        s = skor(cm_dari(y, prediksi(P, aturan, g)), aturan)
        if s > skor_terbaik:
            terbaik, skor_terbaik = g, s
    return round(float(terbaik), 4)


def latih_kalibrator(P, y):
    model = LogisticRegression(C=1e4, max_iter=2000)
    model.fit(np.log(np.clip(P, 1e-6, 1.0)), y)
    return model


def terapkan_kalibrator(model, P):
    return norm(model.predict_proba(np.log(np.clip(P, 1e-6, 1.0))))


KONFIGURASI = [
    ("A", "RF", "rf"), ("A", "XGB", "xgb"),
    ("A", "Hybrid_meta_tanpa_bobot", "hyb_nobal"), ("A", "Hybrid_meta_berbobot", "hyb_bal"),
    ("B", "RF", "rf"), ("B", "XGB", "xgb"),
    ("B", "Hybrid_meta_tanpa_bobot", "hyb_nobal"), ("B", "Hybrid_meta_berbobot", "hyb_bal"),
]

## 4. Menjalankan Evaluasi di Dalam Lipatan

Untuk setiap seed, skenario, model, dan lipatan luar *k*: kalibrator dan parameter aturan dicari dari lipatan lain, lalu dinilai pada lipatan *k*. Tanpa pelatihan ulang sehingga cukup cepat (menit).

In [ ]:
baris = []
stabil = []

for (seed, sk), folds in sorted(GRUP.items()):
    for (sk_k, nama, kunci) in KONFIGURASI:
        if sk_k != sk:
            continue
        Ps = {f: norm(folds[f][kunci]) for f in folds}
        ys = {f: folds[f]["y"].astype(int) for f in folds}

        for f in sorted(folds):
            lain = [g for g in sorted(folds) if g != f]
            P_lain = np.vstack([Ps[g] for g in lain])
            y_lain = np.concatenate([ys[g] for g in lain])

            for kal in [False, True]:
                if kal:
                    kalibrator = latih_kalibrator(P_lain, y_lain)
                    P_cari = terapkan_kalibrator(kalibrator, P_lain)
                    P_nilai = terapkan_kalibrator(kalibrator, Ps[f])
                else:
                    P_cari, P_nilai = P_lain, Ps[f]

                for aturan in RULES:
                    param = cari_param(P_cari, y_lain, aturan)
                    pred = prediksi(P_nilai, aturan, param)
                    rec, pre, f1, gm, mf1, acc = metrik_dari_cm(cm_dari(ys[f], pred))
                    baris.append({
                        "Seed": seed, "Lipatan": f, "Skenario": sk, "Model": nama,
                        "Kalibrasi": "ya" if kal else "tidak", "Aturan": aturan,
                        "Parameter": param if param is not None else np.nan,
                        "Recall_0": rec[0], "Recall_1": rec[1], "Recall_2": rec[2],
                        "Precision_1": pre[1], "F1_1": f1[1],
                        "Macro_F1": mf1, "G_Mean": gm, "Accuracy": acc,
                        "Persen_Prediksi_Kelas1": float((pred == 1).mean() * 100),
                    })
    print("selesai: seed", seed, "skenario", sk)

metrik_aturan = pd.DataFrame(baris)
metrik_aturan.to_csv(os.path.join(OUTPUT_DIR, "metrik_per_lipatan_aturan.csv"), index=False)
print("Baris:", len(metrik_aturan))

## 5. Ringkasan Rata-rata dan Simpangan Baku

Satuan lipatan = satu pasangan (seed, lipatan luar). Simpangan baku antar lipatan (`ddof=1`). **Persen_Prediksi_Kelas1** menunjukkan seberapa banyak orang yang ditandai "waspada prediabetes". Angka yang terlalu besar berarti aplikasi terlalu sering memberi peringatan.

In [ ]:
kunci = ["Skenario", "Model", "Kalibrasi", "Aturan"]
kolom_m = ["Recall_0", "Recall_1", "Recall_2", "Precision_1", "F1_1", "Macro_F1", "G_Mean", "Accuracy",
           "Persen_Prediksi_Kelas1", "Parameter"]

rata = metrik_aturan.groupby(kunci)[kolom_m].mean()
sd = metrik_aturan.groupby(kunci)[kolom_m].std(ddof=1)
n_l = metrik_aturan.groupby(kunci).size().rename("N_Lipatan")
ringkasan = rata.add_suffix("_mean").join(sd.add_suffix("_sd")).join(n_l).reset_index()
ringkasan.to_csv(os.path.join(OUTPUT_DIR, "ringkasan_aturan_mean_sd.csv"), index=False)

tampil_kolom = ["Skenario", "Model", "Kalibrasi", "Aturan", "Recall_1_mean", "Precision_1_mean",
                "Macro_F1_mean", "Macro_F1_sd", "G_Mean_mean", "G_Mean_sd", "Persen_Prediksi_Kelas1_mean"]

print("=" * 110)
print("EFEK ATURAN KEPUTUSAN (tanpa kalibrasi), rata-rata antar lipatan")
print("=" * 110)
print(ringkasan[ringkasan.Kalibrasi == "tidak"][tampil_kolom].round(4).to_string(index=False))

## 6. Apakah Kalibrasi Membantu?

Membandingkan `Kalibrasi = ya` dengan `tidak` pada aturan yang sama. Pada notebook 13 kalibrasi menurunkan G-Mean pada seluruh OOF. Di sini diuji ulang di dalam lipatan.

In [ ]:
pv = ringkasan.pivot_table(index=["Skenario", "Model", "Aturan"], columns="Kalibrasi",
                           values=["G_Mean_mean", "Macro_F1_mean"])
tabel_kal = pd.DataFrame({
    "G_Mean_tanpa_kal": pv[("G_Mean_mean", "tidak")],
    "G_Mean_dengan_kal": pv[("G_Mean_mean", "ya")],
    "Selisih_G_Mean": pv[("G_Mean_mean", "ya")] - pv[("G_Mean_mean", "tidak")],
    "MacroF1_tanpa_kal": pv[("Macro_F1_mean", "tidak")],
    "MacroF1_dengan_kal": pv[("Macro_F1_mean", "ya")],
    "Selisih_MacroF1": pv[("Macro_F1_mean", "ya")] - pv[("Macro_F1_mean", "tidak")],
}).reset_index()
tabel_kal = tabel_kal[tabel_kal.Aturan != "argmax"]
print(tabel_kal.round(4).to_string(index=False))
print()
print("Rata-rata selisih G-Mean akibat kalibrasi (semua konfigurasi dan aturan non-argmax):",
      round(float(tabel_kal["Selisih_G_Mean"].mean()), 4))
print("Rata-rata selisih Macro-F1 akibat kalibrasi:", round(float(tabel_kal["Selisih_MacroF1"].mean()), 4))
tabel_kal.to_csv(os.path.join(OUTPUT_DIR, "efek_kalibrasi.csv"), index=False)

## 7. Stabilitas Parameter

Threshold atau alpha dicari ulang pada tiap lipatan. Jika nilainya bergeser jauh antar lipatan, aturan itu tidak stabil dan angka final akan rapuh. Yang diharapkan: simpangan baku kecil dibanding rata-ratanya.

In [ ]:
stab = (metrik_aturan[metrik_aturan.Aturan != "argmax"]
        .groupby(kunci)["Parameter"].agg(["mean", "std", "min", "max"]).reset_index())
stab.columns = ["Skenario", "Model", "Kalibrasi", "Aturan", "Param_rata2", "Param_sd", "Param_min", "Param_maks"]
stab["Koef_Variasi"] = stab["Param_sd"] / stab["Param_rata2"]
stab.to_csv(os.path.join(OUTPUT_DIR, "stabilitas_parameter.csv"), index=False)
print(stab[stab.Kalibrasi == "tidak"].round(4).to_string(index=False))

## 8. Perbandingan Berpasangan Antar Aturan

Selisih pada **lipatan yang sama**, dengan uji t terkoreksi Nadeau dan Bengio (rasio uji/latih = 1/(`N_OUTER` - 1)) dan Wilcoxon. Dengan 5 lipatan nilai p Wilcoxon terkecil 0,0625, jadi perhatikan juga besar selisih dan konsistensi tanda (`Lipatan_Positif`). Selisih = konfigurasi pertama dikurangi kedua.

In [ ]:
def uji_pasangan(selisih, n_outer):
    selisih = np.asarray(selisih, dtype=float)
    selisih = selisih[~np.isnan(selisih)]
    J = len(selisih)
    if J < 2:
        return dict(N=J, Rata_Selisih=np.nan, SD_Selisih=np.nan, t_terkoreksi=np.nan, p_t=np.nan,
                    p_wilcoxon=np.nan, Lipatan_Positif=np.nan)
    rata_s = selisih.mean()
    var = selisih.var(ddof=1)
    if var == 0:
        t_stat, p_t = (np.inf if rata_s != 0 else 0.0), (0.0 if rata_s != 0 else 1.0)
    else:
        t_stat = rata_s / np.sqrt((1 / J + 1 / (n_outer - 1)) * var)
        p_t = 2 * stats.t.sf(abs(t_stat), df=J - 1)
    try:
        p_w = stats.wilcoxon(selisih).pvalue
    except ValueError:
        p_w = np.nan
    return dict(N=J, Rata_Selisih=rata_s, SD_Selisih=np.sqrt(var), t_terkoreksi=t_stat, p_t=p_t,
                p_wilcoxon=p_w, Lipatan_Positif=int((selisih > 0).sum()))


PASANGAN_ATURAN = [
    ("thr_gmean vs argmax", ("tidak", "thr_gmean"), ("tidak", "argmax")),
    ("alpha_macrof1 vs argmax", ("tidak", "alpha_macrof1"), ("tidak", "argmax")),
    ("thr_macrof1 vs alpha_macrof1", ("tidak", "thr_macrof1"), ("tidak", "alpha_macrof1")),
    ("thr_gmean vs alpha_gmean", ("tidak", "thr_gmean"), ("tidak", "alpha_gmean")),
    ("kalibrasi + thr_gmean vs thr_gmean", ("ya", "thr_gmean"), ("tidak", "thr_gmean")),
    ("kalibrasi + alpha_macrof1 vs alpha_macrof1", ("ya", "alpha_macrof1"), ("tidak", "alpha_macrof1")),
]
METRIK_UJI = ["G_Mean", "Macro_F1", "Recall_1", "Precision_1"]

hasil_uji = []
for (sk, nama, _) in KONFIGURASI:
    sub = metrik_aturan[(metrik_aturan.Skenario == sk) & (metrik_aturan.Model == nama)]
    for nama_p, (k1, a1), (k2, a2) in PASANGAN_ATURAN:
        a = sub[(sub.Kalibrasi == k1) & (sub.Aturan == a1)].set_index(["Seed", "Lipatan"])
        b = sub[(sub.Kalibrasi == k2) & (sub.Aturan == a2)].set_index(["Seed", "Lipatan"])
        gab = a.join(b, lsuffix="_1", rsuffix="_2", how="inner")
        for metrik in METRIK_UJI:
            r = uji_pasangan((gab[metrik + "_1"] - gab[metrik + "_2"]).values, N_OUTER)
            r.update(Skenario=sk, Model=nama, Pasangan=nama_p, Metrik=metrik)
            hasil_uji.append(r)

tabel_uji = pd.DataFrame(hasil_uji)[["Skenario", "Model", "Pasangan", "Metrik", "N", "Rata_Selisih", "SD_Selisih",
                                      "Lipatan_Positif", "t_terkoreksi", "p_t", "p_wilcoxon"]]
tabel_uji.to_csv(os.path.join(OUTPUT_DIR, "uji_aturan_berpasangan.csv"), index=False)

ringkas_uji = tabel_uji[tabel_uji.Metrik == "G_Mean"].copy()
for kolom in ["Rata_Selisih", "SD_Selisih"]:
    ringkas_uji[kolom] = ringkas_uji[kolom].map(lambda v: f"{v:+.4f}" if np.isfinite(v) else "-")
for kolom in ["p_t", "p_wilcoxon"]:
    ringkas_uji[kolom] = ringkas_uji[kolom].map(lambda v: f"{v:.3f}" if np.isfinite(v) else "-")
print("Selisih G-Mean (semua metrik ada di uji_aturan_berpasangan.csv):")
print(ringkas_uji[["Skenario", "Model", "Pasangan", "Rata_Selisih", "Lipatan_Positif", "p_t", "p_wilcoxon"]].to_string(index=False))

## 9. Ranking Kandidat dan Usulan Konfigurasi Final

Semua kombinasi diurutkan menurut kriteria utama (`KRITERIA_PILIH`). Karena banyak kombinasi hampir setara, dipakai **aturan satu galat baku**:

1. Cari kombinasi dengan rata-rata terbaik.
2. Hitung galat bakunya: `sd / akar(jumlah lipatan)`.
3. Semua kombinasi yang rata-ratanya berada dalam satu galat baku dari yang terbaik dianggap setara.
4. Dari yang setara, pilih yang **paling sederhana**. Skor kompleksitas = model (RF dan XGBoost 0, Hybrid tanpa bobot 1, Hybrid berbobot 2) + aturan (`argmax` 0, threshold atau alpha 1) + kalibrasi (ya 1). Jika kompleksitas sama, pilih rata-rata tertinggi.

Cara ini mencegah memilih model rumit hanya karena unggul tipis yang masih dalam kisaran derau antar lipatan. **Hasilnya adalah usulan.** Keputusan akhir (termasuk apakah Hybrid dipertahankan sebagai klaim judul) tetap dibahas dengan pembimbing.

In [ ]:
KOMPLEKSITAS_MODEL = {"RF": 0, "XGB": 0, "Hybrid_meta_tanpa_bobot": 1, "Hybrid_meta_berbobot": 2}

rk = ringkasan.copy()
rk["Kompleksitas"] = (rk["Model"].map(KOMPLEKSITAS_MODEL)
                      + (rk["Aturan"] != "argmax").astype(int)
                      + (rk["Kalibrasi"] == "ya").astype(int))
rk = rk[rk["Recall_1_mean"] >= SYARAT_RECALL_1].copy()
m_kol, s_kol = KRITERIA_PILIH + "_mean", KRITERIA_PILIH + "_sd"
rk = rk.sort_values(m_kol, ascending=False).reset_index(drop=True)
rk.insert(0, "Peringkat", np.arange(1, len(rk) + 1))
rk.to_csv(os.path.join(OUTPUT_DIR, "ranking_kandidat.csv"), index=False)

terbaik = rk.iloc[0]
galat_baku = terbaik[s_kol] / np.sqrt(terbaik["N_Lipatan"])
ambang = terbaik[m_kol] - galat_baku
setara = rk[rk[m_kol] >= ambang].sort_values(["Kompleksitas", m_kol], ascending=[True, False])
usulan = setara.iloc[0]

kolom_lihat = ["Peringkat", "Skenario", "Model", "Kalibrasi", "Aturan", "Recall_1_mean", "Precision_1_mean",
               "Macro_F1_mean", "G_Mean_mean", "G_Mean_sd", "Persen_Prediksi_Kelas1_mean", "Kompleksitas"]
print(f"Kriteria utama: {KRITERIA_PILIH} | galat baku kandidat terbaik: {galat_baku:.4f} | ambang setara: {ambang:.4f}")
print()
print("10 kandidat teratas menurut rata-rata:")
print(rk.head(10)[kolom_lihat].round(4).to_string(index=False))
print()
print(f"Kandidat setara (dalam satu galat baku): {len(setara)}")
print(setara.head(10)[kolom_lihat].round(4).to_string(index=False))
print()
print("USULAN (paling sederhana di antara yang setara):")
print(usulan[kolom_lihat].to_string())

In [ ]:
# Parameter aturan untuk usulan, dicari dari SELURUH probabilitas lipatan luar (satu seed pertama yang lengkap).
# Angka ini hanya informasi. Notebook 17 mencarinya lagi dengan cara yang sama sebelum dipakai.
sk_u, model_u = usulan["Skenario"], usulan["Model"]
kunci_u = [k for (s, n, k) in KONFIGURASI if s == sk_u and n == model_u][0]
seed_u = sorted(s for (s, sk_) in GRUP if sk_ == sk_u)[0]
P_semua = np.vstack([norm(GRUP[(seed_u, sk_u)][f][kunci_u]) for f in sorted(GRUP[(seed_u, sk_u)])])
y_semua = np.concatenate([GRUP[(seed_u, sk_u)][f]["y"].astype(int) for f in sorted(GRUP[(seed_u, sk_u)])])
if usulan["Kalibrasi"] == "ya":
    P_semua = terapkan_kalibrator(latih_kalibrator(P_semua, y_semua), P_semua)
param_u = cari_param(P_semua, y_semua, usulan["Aturan"])

kandidat = {
    "CATATAN": "USULAN otomatis dari notebook 16, bukan keputusan final. Dibahas dengan pembimbing sebelum notebook 17.",
    "skenario": sk_u,
    "model": model_u,
    "kalibrasi": usulan["Kalibrasi"] == "ya",
    "aturan": usulan["Aturan"],
    "parameter_dari_seluruh_oof": param_u,
    "kriteria_pilih": KRITERIA_PILIH,
    "estimasi_cv_dalam_lipatan": {
        "Macro_F1_mean": float(usulan["Macro_F1_mean"]), "Macro_F1_sd": float(usulan["Macro_F1_sd"]),
        "G_Mean_mean": float(usulan["G_Mean_mean"]), "G_Mean_sd": float(usulan["G_Mean_sd"]),
        "Recall_1_mean": float(usulan["Recall_1_mean"]), "Precision_1_mean": float(usulan["Precision_1_mean"]),
        "N_Lipatan": int(usulan["N_Lipatan"]),
    },
    "uji_cepat": UJI_CEPAT,
}
with open(os.path.join(OUTPUT_DIR, "kandidat_final.json"), "w", encoding="utf-8") as f:
    json.dump(kandidat, f, indent=2, ensure_ascii=False)
print(json.dumps(kandidat, indent=2, ensure_ascii=False))

## 10. Grafik

In [ ]:
WARNA_ATURAN = {"argmax": "#8A8A8A", "thr_gmean": "#4C78A8", "thr_macrof1": "#72A5D4",
                "alpha_macrof1": "#E08E45", "alpha_gmean": "#F2B880"}

fig, ax = plt.subplots(1, 2, figsize=(17, 5.6))

teratas = rk.head(12).iloc[::-1]
etiket = [f"{r.Skenario}-{r.Model.replace('Hybrid_meta_', 'Hyb ')} | {r.Aturan} | kal:{r.Kalibrasi}" for r in teratas.itertuples()]
ax[0].barh(range(len(teratas)), teratas[m_kol], xerr=teratas[s_kol].fillna(0),
           color=[WARNA_ATURAN[a] for a in teratas["Aturan"]], capsize=2)
ax[0].set_yticks(range(len(teratas)))
ax[0].set_yticklabels(etiket, fontsize=7)
ax[0].set_xlabel(KRITERIA_PILIH + " (rata-rata, batang galat = SD antar lipatan)")
ax[0].set_title("12 kandidat teratas")

for aturan, warna in WARNA_ATURAN.items():
    s = rk[(rk.Aturan == aturan) & (rk.Kalibrasi == "tidak")]
    ax[1].scatter(s["Recall_1_mean"], s["Precision_1_mean"], color=warna, label=aturan, s=45)
ax[1].set_xlabel("Recall Prediabetes (rata-rata)")
ax[1].set_ylabel("Presisi Prediabetes (rata-rata)")
ax[1].set_title("Kompromi recall dan presisi (tanpa kalibrasi)")
ax[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "grafik_aturan.png"), dpi=150, bbox_inches="tight")
plt.show()

## 11. Catatan Akhir

In [ ]:
print("Hasil tersimpan di:", OUTPUT_DIR)
for nama in sorted(os.listdir(OUTPUT_DIR)):
    print("  -", nama)
print("\nCATATAN:")
print("- Tidak ada pelatihan. train_final.csv dan test_final_locked.csv TIDAK dibaca. Folder artifacts dan data TIDAK diubah.")
print("- kandidat_final.json adalah USULAN. Bahas dengan pembimbing sebelum menjalankan notebook 17.")
if UJI_CEPAT:
    print("- MODE UJI CEPAT: hasil ini BUKAN untuk penelitian.")

## 12. Cara Membaca Hasil untuk Laporan

1. **Aturan keputusan:** bandingkan `thr_gmean` dan `alpha_macrof1` dengan `argmax` pada konfigurasi yang sama (`uji_aturan_berpasangan.csv`). Jika threshold dan alpha setara, aturan yang lebih sederhana dan sudah dipakai aplikasi (threshold) cukup, lalu jelaskan di Bab III bahwa rumusan alpha pada proposal diganti beserta alasan dan buktinya.
2. **Kalibrasi:** jika `Selisih_G_Mean` rata-rata negatif atau tidak konsisten, simpulkan bahwa kalibrasi tidak memperbaiki keputusan (walaupun mungkin memperbaiki Log Loss, yang tidak dipakai untuk memilih).
3. **Persen prediksi kelas 1:** laporkan jujur bahwa aturan terpilih menandai sekian persen orang sebagai waspada dengan presisi rendah. Ini sifat skrining sensitif, bukan diagnosis.
4. **Hybrid vs model tunggal:** pilihan satu galat baku memihak model sederhana. Jika yang terpilih adalah model tunggal, klaim Hybrid pada judul perlu dilunakkan atau diganti, dan itu keputusan yang dibahas dengan pembimbing.
5. **Keterbatasan:** parameter aturan dicari dari lipatan lain pada data latih yang sama dengan pemilihan hiperparameter, jadi estimasi tetap sedikit optimistis. Estimasi tak bias baru datang dari uji terkunci di notebook 17.